In [1]:
import torch
from transformers import pipeline

# Sistemimizdeki RTX 5000 Ada GPU'yu seçiyoruz
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Kullanılan cihaz: {device}")

# Whisper modelini indirip ekran kartımızın hafızasına yüklüyoruz
print("Whisper modeli indiriliyor... (İlk seferde biraz sürebilir)")
asr_model = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=device
)
print("✅ Model başarıyla yüklendi ve dinlemeye hazır!")

Kullanılan cihaz: cuda:0
Whisper modeli indiriliyor... (İlk seferde biraz sürebilir)


Device set to use cuda:0


✅ Model başarıyla yüklendi ve dinlemeye hazır!


In [2]:
import urllib.request
import os
import time

# 1. Aşama: Test sesini indiriyoruz (Eğer zaten yoksa)
audio_url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac"
audio_path = "test_audio.flac"

if not os.path.exists(audio_path):
    print("Test sesi internetten indiriliyor...")
    urllib.request.urlretrieve(audio_url, audio_path)
    print("✅ İndirme tamamlandı!")

print("🎧 Whisper modeli o güçlü GPU'da sesi çözümlüyor, bekle...")

# 2. Aşama: Kronometreyi başlatıp sesi modele veriyoruz
baslangic = time.time()

# Sadece dosyanın yolunu modele vermemiz yeterli, o ne yapacağını biliyor
sonuc = asr_model(audio_path)

bitis = time.time()
gecen_sure = bitis - baslangic

# 3. Aşama: Sonuçları ekrana şık bir şekilde basıyoruz
print("\n" + "="*50)
print("🗣️ ÇIKARILAN İNGİLİZCE METİN:")
print("="*50)
print(sonuc["text"].strip())
print("="*50)
print(f"⏱️ Bu işlem senin RTX 5000'inde sadece {gecen_sure:.2f} saniye sürdü!")

🎧 Whisper modeli o güçlü GPU'da sesi çözümlüyor, bekle...


`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.



🗣️ ÇIKARILAN İNGİLİZCE METİN:
He hoped there would be stew for dinner, turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick, peppered, flour-fattened sauce.
⏱️ Bu işlem senin RTX 5000'inde sadece 1.44 saniye sürdü!


In [3]:
from datasets import load_dataset
import os

print("Toplu veri seti indiriliyor...")

# Hugging Face'ten test amaçlı küçük bir İngilizce konuşma veri seti çekiyoruz
# 'PolyAI/minds14' küçük ve hızlı olduğu için testlerde çok kullanışlıdır
# 'en-US' (Amerikan İngilizcesi) kısmını seçiyoruz
dataset = load_dataset("PolyAI/minds14", name="en-US", split="train")

# Testleri hızlı yapmak için sadece ilk 5 ses dosyasını seçeceğiz. 
# İstersen bu sayıyı 200 yapabilirsin!
test_edilecek_ses_sayisi = 5
kucuk_veri_seti = dataset.select(range(test_edilecek_ses_sayisi))

print(f"✅ {test_edilecek_ses_sayisi} adet ses dosyası başarıyla yüklendi ve teste hazır!")

Toplu veri seti indiriliyor...
✅ 5 adet ses dosyası başarıyla yüklendi ve teste hazır!


DATASETTEKİ SESLERİ OYNATMAK İÇİN KOD:

In [4]:
from IPython.display import Audio, display

print("🎧 İlk ses dosyası yükleniyor...")

# Veri setimizden ilk sesi (0. indeks) alıyoruz
ilk_ornek = kucuk_veri_seti[3]

# Sesin sayısal dizisini ve frekansını (kalitesini) çekiyoruz
ses_dizisi = ilk_ornek["audio"]["array"]
ornekleme_hizi = ilk_ornek["audio"]["sampling_rate"]

# Jupyter içinde tıklanabilir bir ses oynatıcı oluşturuyoruz
display(Audio(data=ses_dizisi, rate=ornekleme_hizi))

print("Gerçekte Söylenen:", ilk_ornek['english_transcription'])

🎧 İlk ses dosyası yükleniyor...


/home/ceren/miniconda3/envs/voice_pipeline/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Gerçekte Söylenen: how do I start a joint account


In [5]:
import time

print("="*60)
print("🚀 OTOMATİK TOPLU TEST BAŞLIYOR")
print("="*60)

toplam_sure = 0

# Veri setindeki her bir ses dosyası için bir döngü başlatıyoruz
for index, ornek in enumerate(kucuk_veri_seti):
    
    # Veri setinden doğrudan sesin sayısal dizisini ve frekansını alıyoruz
    ses_dizisi = ornek["audio"]["array"]
    
    baslangic = time.time()
    
    # Sesi Whisper'a veriyoruz
    sonuc = asr_model(ses_dizisi)
    
    bitis = time.time()
    gecen_sure = bitis - baslangic
    toplam_sure += gecen_sure
    
    # Çıktıları ekrana şık bir şekilde basıyoruz
    print(f"\n🎧 DOSYA {index + 1}:")
    print(f"Gerçek Metin (Veri setindeki asıl kayıt): {ornek['english_transcription']}")
    print(f"Whisper'ın Duyduğu: {sonuc['text'].strip()}")
    print(f"⏱️ Çözüm Süresi: {gecen_sure:.2f} saniye")
    print("-" * 40)

print("\n" + "="*60)
print(f"✅ TEST TAMAMLANDI!")
print(f"Toplam {test_edilecek_ses_sayisi} dosya için harcanan süre: {toplam_sure:.2f} saniye")
print("="*60)

🚀 OTOMATİK TOPLU TEST BAŞLIYOR

🎧 DOSYA 1:
Gerçek Metin (Veri setindeki asıl kayıt): I would like to set up a joint account with my partner
Whisper'ın Duyduğu: I would like to set up a joint account with my partner. How do I proceed with doing that?
⏱️ Çözüm Süresi: 0.49 saniye
----------------------------------------

🎧 DOSYA 2:
Gerçek Metin (Veri setindeki asıl kayıt): Henry County set up a joint account with my wife and where are they at
Whisper'ın Duyduğu: Please write it down in the comments if you want to take my life and where the app might be.
⏱️ Çözüm Süresi: 0.48 saniye
----------------------------------------

🎧 DOSYA 3:
Gerçek Metin (Veri setindeki asıl kayıt): hi I'd like to set up a joint account with my partner I'm not seeing the option to do it on the app so I called in to get some help can I do it over the phone with you and give you the information
Whisper'ın Duyduğu: Hi, I'd like to set up a joint account with my partner. I'm not seeing the option to do it on the app

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("Qwen modeli indiriliyor ve GPU'ya yükleniyor. Bu biraz sürebilir...")

# Qwen'in hızlı ve çok akıllı 1.5 Milyar parametrelik versiyonu
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. Qwen'in Sözlüğünü indiriyoruz
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 2. Asıl beyni indirip doğrudan RTX 5000 ekran kartına yüklüyoruz
# Hafızayı optimum kullanmak için bfloat16 formatını seçiyoruz
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)

print("✅ Qwen modeli başarıyla yüklendi ve çeviriye hazır!")

Qwen modeli indiriliyor ve GPU'ya yükleniyor. Bu biraz sürebilir...


`torch_dtype` is deprecated! Use `dtype` instead!


✅ Qwen modeli başarıyla yüklendi ve çeviriye hazır!


In [7]:
import torch

print("🧠 Qwen modeli katı kurallarla tekrar çalışıyor...")

# Aynı hatalı 4. indeksi alıyoruz
ornek = kucuk_veri_seti[2] 
whisper_hatali_cikti = asr_model(ornek["audio"]["array"])["text"].strip()

# ÇOK DAHA KATI BİR PROMPT (Sistem Komutu)
mesajlar = [
    {"role": "system", "content": "Sen sadece İngilizceden Türkçeye çeviri yapan bir makinesin. Gelen metin mantıksız olsa bile sadece en yakın anlamıyla Türkçeye çevir. ASLA açıklama yapma. ASLA yorum yapma. Sadece çeviriyi ver."},
    {"role": "user", "content": f"{whisper_hatali_cikti}"}
]

# Mesajı Qwen'in anlayacağı formata çevirip GPU'ya yolluyoruz
prompt = tokenizer.apply_chat_template(mesajlar, tokenize=False, add_generation_prompt=True)
girdiler = tokenizer(prompt, return_tensors="pt").to("cuda")

# PARAMETRE AYARI: temperature=0.1 ve do_sample=True ile halüsinasyonu bitiriyoruz
with torch.no_grad():
    ciktilar = llm_model.generate(
        **girdiler, 
        max_new_tokens=100,
        temperature=0.1,  
        do_sample=True,
        repetition_penalty=1.1 
    )

qwen_cevabi = tokenizer.decode(ciktilar[0][girdiler.input_ids.shape[-1]:], skip_special_tokens=True)

print("\n" + "="*60)
print(f"🎯 GERÇEKTE SÖYLENEN : {ornek['transcription']}")
print(f"❌ WHISPER'IN DUYDUĞU : {whisper_hatali_cikti}")
print(f"✨ QWEN'İN ÇEVİRİSİ  : {qwen_cevabi.strip()}")
print("="*60)

🧠 Qwen modeli katı kurallarla tekrar çalışıyor...

🎯 GERÇEKTE SÖYLENEN : hi I'd like to set up a joint account with my partner I'm not seeing the option to do it on the app so I called in to get some help can I do it over the phone with you and give you the information
❌ WHISPER'IN DUYDUĞU : Hi, I'd like to set up a joint account with my partner. I'm not seeing the option to do it on the app, so I called him to get some help. Can I just do it over the phone with you and give you the information? Or should I do it on the app and I'm missing something? Okay, I'd prefer to just do it over the phone if possible. Thanks.
✨ QWEN'İN ÇEVİRİSİ  : Merhaba, evliğimde bir ortak hesap oluşturmak istiyorum. Umarım uygulamanın arka planında bu seçenek buluyor, ancak telefonla yardımcı olabilir miyim? Size bilgi verebilir misiniz? Eğer uygulama üzerinden yapabiliyorsanız ne hata yaptığınızı bana anlatabilirim? Telefonla yapmak isterdim, mümkünse. Teşekkür ed


In [15]:
import os
import torch
from TTS.api import TTS

# Coqui'nin kullanım koşullarını otomatik olarak kabul ediyoruz
os.environ["COQUI_TOS_AGREED"] = "1"

print("🎙️ XTTS-v2 Ses motoru indiriliyor ve GPU'ya yükleniyor...")

# RTX 5000'i tanımlıyoruz
device = "cuda" if torch.cuda.is_available() else "cpu"

# Modeli indirip VRAM'e park ediyoruz
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

print("✅ XTTS başarıyla yüklendi ve konuşmaya hazır!")

🎙️ XTTS-v2 Ses motoru indiriliyor ve GPU'ya yükleniyor...
 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.


ImportError: cannot import name 'BeamSearchScorer' from 'transformers' (/home/ceren/.local/lib/python3.10/site-packages/transformers/__init__.py)